# Tutorial: A Competitive Bidding Round with OpenArcade, XChange and AgentGrid
- Author: Shridhar Kini
- To Securely Run: `jupyter notebook password` to generate onetime password for secure access
- To Run at root directory of the repo: `jupyter notebook --allow-root --port 9999 --ip=0.0.0.0`
- To Clear Outputs: Use `jupyter nbconvert --clear-output --inplace Video_Analytics_Bidding.ipynb`

**A real tender, start to finish:**
* A city publishes an RFP for a video analytics system
* 5 vendor companies bid
* The evaluator scores them on 4 dimensions
* Highest total wins
* Nothing is scripted. The agents read the RFP and decide for themselves.

**Each company has 6 agents:**

| Agent | What it does |
| :--- | :--- |
| **Bid Manager** | Runs the other five, submits or declines the bid |
| **AI Compliance Agent** | Checks RFP requirements against the catalogue, offers live endpoints |
| **Sizing Agent** | Works out hardware for the licence counts |
| **Finance Agent** | Prices the bid |
| **Bid Reviewer Agent** | Checks the bid is complete and consistent |
| **Head Agent** | Approves or refuses, based on margin |

**Scale:** 5 companies x 6 agents = 30 pods, 4 functions, 1 evaluator.

The companies are deliberately unequal, so some qualify, some decline, and one wins.

# The three systems this example sits on

```
                         XChange  (the marketplace)
                            |  a buyer posts one task
                            |  five vendors are resolved: 3 named, 2 by topic
                            v
                      OpenArcade  (the bidding system)
                            |  calls the pre-qualification function per bid
                            |  calls the evaluator once, when the last bid lands
                            v
                        AgentGrid  (the 30 agents)
                            |  OpenMesh carries agent-to-agent messages
                            |  HIS records what every agent received and produced
```

| System | What it does here |
| :--- | :--- |
| **XChange** | Buyer posts one task. 3 companies invited by name, 2 join through the topic `videoanalytics_bidding` |
| **OpenArcade** | Runs the bid job. Calls PQT per bid, and the evaluator once when all bids are in (60s, no retry) |
| **ServiceGrid Functions** | 4 kinds: PQT, evaluator, RFP requirements, per-company live endpoints |
| **HIS** | Stores what each agent received and decided. The dashboard reads it |

## 1. Directory Structure

* **`nodes/`** - agent code, one folder per company
  * Each company has 6 agents and a `config.yaml` (catalogue, prices, credentials, margin)
  * `nodes/common/` is shared code
  * `nodes/local_check.py` tests without a cluster
* **`spec/`** - 30 agent specs, register scripts, `register_subjects_in_Exchange.sh`
* **`functions/`** - 4 functions, their scripts and tests
* **`deploy_run/`** - this notebook, deploy scripts, seeders, `va_bidding_request.sh`, dashboard

**To change who wins, edit `config.yaml`, not code.**

## 2. Prerequisites

1. `.env` at the repo root, from `.env.template`
   * Generic settings only
   * Example options (tender, dashboard port, timeouts) have defaults in the scripts
2. `HIS_BASE_URL` set. Without it the round runs, but the dashboard is mostly empty.
3. Cluster running: XChange, OpenArcade, function registry, MinIO, HIS
4. A valid `OPENAI_API_KEY` in `.env`
   * every agent runs `openai:gpt-5.4-mini`
   * `register.sh` copies the key into all 30 specs, and a pod reads its spec only at start-up
   * a bad key means every company declines at qualification, so `register.sh` checks it first
5. 3 MinIO buckets set to anonymous download (step 3.1)

Check the settings:

In [ ]:
%%bash
(
  cd ../../..
  set -a; source .env; set +a
  # These five must be set; the round cannot reach the cluster without them.
  for v in EXCHANGE_BASE_URL OPENARCADE_BIDDING_URL FUNCTION_REGISTRY_URL \
           FUNCTION_UPLOAD_URL HIS_BASE_URL; do
    printf '  %-24s %s\n' "$v" "${!v:-*** REQUIRED, NOT SET ***}"
  done
  printf '  %-24s %s\n' OPENAI_API_KEY "$([ -n "${OPENAI_API_KEY:-}" ] && echo set || echo '*** REQUIRED, NOT SET ***')"
  printf '  %-24s %s\n' FUNCTION_EXECUTOR_ID "${FUNCTION_EXECUTOR_ID:-executor-001 (default)}"
  # Not in .env.template -- an override with a working default, shown so an unset value
  # does not look broken.
  printf '  %-24s %s\n' VA_STREAMLIT_PORT "${VA_STREAMLIT_PORT:-8008 (default)}"
)

## 3. Seed the tender and the image set

Two things must be in MinIO before a round:

**RFP** - a real smart-city tender extract
* Default is `patna`. Use `RFP=chennai` for the other.
* Different RFPs have different requirement counts
* Inside one round, every company is scored out of the same number

**Evaluation image set** - same images and ground truth for every bidder
* Ground truth travels with the bid job, so anyone can check the scoring by hand

Seeders skip files already in MinIO. Use `SEED_FORCE=1` to re-upload.

In [ ]:
%%bash
(
  cd ..
  set -a; source ../../.env; set +a
  # RFP=chennai and SEED_FORCE=1 go here, in front of the command, when you want them.
  ../../venv/bin/python deploy_run/seed_rfp.py
  ../../venv/bin/python deploy_run/seed_eval_images.py
)

### 3.1. Make the three buckets download-only

**Manual step, once per cluster.** The seeders create the buckets. This sets who can read them.

| Bucket | Holds |
| :--- | :--- |
| `va-bidding-rfp` | the tender |
| `va-bidding-eval` | the evaluation images |
| `va-bidding-docs` | each company's commercial and sizing workbooks |

In the MinIO Console, for each bucket:

```
Buckets -> <bucket> -> Anonymous -> Add Access Rule
    Prefix:  /
    Access:  readonly
```

**Why:**
* Documents are opened by URL
* Readers (dashboard, reviewers) have no MinIO credentials
* Without this, uploads work but every link shows 403

In [ ]:
%%bash
(
  cd ../../..
  set -a; source .env; set +a
  MINIO="http://${VA_MINIO_EXTERNAL_URL:-$EXTERNAL_IP:$MINIO_EXTERNAL_PORT}"
  echo "MinIO S3 endpoint: $MINIO   (the console is a separate port on the same host)"
  for b in va-bidding-rfp va-bidding-eval va-bidding-docs; do
    listing=$(curl -s --max-time 15 "$MINIO/$b/?list-type=2&max-keys=1")
    key=$(printf '%s' "$listing" | grep -o '<Key>[^<]*' | head -1 | cut -c6-)
    if [ -z "$key" ]; then
      printf '  %-18s %s\n' "$b" "no objects yet (or listing is blocked)"
      continue
    fi
    code=$(curl -s --max-time 15 -o /dev/null -w '%{http_code}' "$MINIO/$b/$key")
    case "$code" in
      200) printf '  %-18s readable  (%s)\n' "$b" "$key" ;;
      403) printf '  %-18s BLOCKED -- add the anonymous readonly rule\n' "$b" ;;
      *)   printf '  %-18s HTTP %s on %s\n' "$b" "$code" "$key" ;;
    esac
  done
)

## 4. Build and deploy the four functions

| Function | Called by | Purpose |
| :--- | :--- | :--- |
| `va-bidding-pqt` | OpenArcade, per bid | Pre-qualification: certifications, licences, projects |
| `va-bid-eval` | OpenArcade, once per round | Scores bids and picks the winner |
| `va-rfp-requirements` | the 5 AI Compliance Agents | Reads the RFP once, gives everyone the same requirement list |
| `va-uc-{company}-{usecase}` | the evaluator | Company's live endpoint for the bake-off (10 in total) |

**Why `va-rfp-requirements` matters:**
* Before: each company extracted its own list, so totals differed (0/16 vs 1/70)
* Now: one function extracts once and caches it
* Everyone is scored out of the same total

In [ ]:
%%bash
(
  cd ../functions
  bash build.sh
)

### 4.1. Upload them to the registry

`upload.sh` uploads all functions. Pass names to upload only some.

In [ ]:
%%bash
(
  cd ../functions
  bash upload.sh
)

### 4.2. Warm the deployments

**Do not skip this.**
* A new function has no running deployment
* The first call has to start a container (takes minutes)
* OpenArcade gives the evaluator 60s and does not retry
* `warmup.sh` starts all 13 and waits until each one answers
* It also prints how to create any missing company answer files

> **Note on `kubectl`:**
> * `KUBECTL_COMMAND` in `.env` is an alias, which notebooks cannot use
> * Cells call `kubectl` directly
> * Need a kubeconfig? Run `export KUBECONFIG=~/.kube/<your-config>` before starting Jupyter

In [ ]:
%%bash
(
  cd ../functions
  bash warmup.sh
)

In [ ]:
!kubectl get pods -n policies | grep -E 'NAME|va-'

## 5. Register the 30 agents and the 5 companies

**Two separate registrations:**
* `register.sh` - the 30 agent specs
* `register_subjects_in_Exchange.sh` - the 5 Bid Managers as XChange subjects (2 carry the topic `videoanalytics_bidding`)

**Why both are needed:**
* The buyer names 3 companies
* The other 2 join only through the topic
* Skip the second script and the round runs with 3 bidders, with no error

**The topic is in two places:**
* XChange subject: `topics` (how XChange finds the 2 companies)
* Agent spec: `subject_metadata.topics` (what the Bid Manager polls OpenArcade for)
* Both come from `company.topics` in the company's `config.yaml`, so they cannot disagree
* Search tags are not topics. They only help find agents, they do not tell an agent what to listen to.

In [ ]:
%%bash
(
  cd ../spec
  bash register.sh
)

In [ ]:
%%bash
(
  cd ../spec
  bash register_subjects_in_Exchange.sh
)

### 5.1. What an XChange subject holds

| Field | Purpose |
| :--- | :--- |
| `subject_id` | Name the buyer invites by |
| `subject_metadata` | Company info, for display |
| `subject_capabilities` | Use cases, licence range, certifications. Can be queried. |
| `topics` | Subscriptions. A task with a topic reaches every subject holding it. |
| `task_evaluation_function` | Only for `open` mode. Empty here. |

* Built from each company's `config.yaml`
* To change a topic, edit the config, not the script

Read them back. The last two have the topic:

In [ ]:
%%bash
(
  cd ../../..
  set -a; source .env; set +a
  # jq -r prints each comma-separated value on its own line, so the layout needs no
  # escapes of its own.
  for s in camfacesolution multifacetech newgentech ultravideotech videoproctech; do
    curl -s --max-time 15 "$EXCHANGE_BASE_URL/subjects/${s}-bid-manager" | jq -r '
      (.data // .) |
      ( "  \(.subject_id)",
        "      topics      \(if (.topics // []) == [] then "(none -- invited by name only)" else (.topics | join(", ")) end)",
        "      usecases    \((.subject_capabilities.usecases // []) | join(", "))",
        "      licences    \(.subject_capabilities.min_licenses // "?") to \(.subject_capabilities.max_licenses // "?")" )'
  done
)

## 6. Build and push the agent images

* One image per agent, tagged with its `subject_id`
* That tag must match the `AGENT` value in `entrypoint.sh` and the name in the spec
* 30 images take a while
* Script also accepts a company name, `managers`, or one agent id

In [ ]:
%%bash
(
  cd ../..
  bash build_va_tenderprocessing_agents_and_push.bash
)

## 7. Deploy the 30 agent pods

In [ ]:
%%bash
(
  cd ../deploy_run
  bash deploy.sh
)

### Verify that the agents are running

* Wait for all 30 to be `Running`
* `Running` does not mean ready. Agents need a moment to join the mesh.

In [ ]:
!kubectl get pods -n agents | grep -E 'NAME|executor-' | head -35

## 8. Run a bidding round

The buyer does one thing: **posts one task.**

**What happens:**
1. Task resolves to 5 bidders (3 named, 2 by topic)
2. Each Bid Manager calls its 5 sub-agents in order
3. All 5 compliance agents call `va-rfp-requirements` (one extracts, four use the cache)
4. Each company submits or declines. OpenArcade runs PQT on every bid.
5. The last bid triggers the evaluator
6. The evaluator calls each company's live endpoints and scores them against the ground truth

### 8.1. The one task

XChange has 4 assignment modes. This example uses `bidding`.

| Mode | How the subject is chosen |
| :--- | :--- |
| `direct` | You name one subject |
| `function` | A registry function picks one |
| `open` | Each subject's own function is asked until one accepts |
| `bidding` | A bid job opens and companies compete (**used here**) |

**Bid job mode** (OpenArcade). This example uses `mixed`.

| Mode | Who can bid |
| :--- | :--- |
| `closed` | Only the subjects named in `bid_job_subject_ids` |
| `open` | Anyone who applies, until the job is full or time runs out |
| `mixed` | Named subjects **and** subjects found by topic (a fixed roster, also discoverable by tag) |

```json
{
  "task_assignment_type": "bidding",
  "task_data": { "rfp": {"bucket": "...", "object": "...", "url": "..."},
                 "rfp_name": "...", "project_type": "video_analytics" },
  "task_metadata": {
    "task_assignment": {
      "bid_job_evaluator_id": "va-bid-eval:1.3-stable",
      "bid_job_pqt_id":       "va-bidding-pqt:1.2-stable",
      "bid_job_mode":         "mixed",
      "bid_job_subject_ids":  ["camfacesolution-bid-manager", "...", "..."],
      "topics":               ["videoanalytics_bidding"],
      "bid_job_tags":         ["videoanalytics_bidding"],
      "bid_job_description":  {"task_type": "video_analytics_rfp", "rfp_url": "...", "text": "..."},
      "bid_job_metadata":     {"evaluation": {"sample_images": [...], "weights": {...}}}
    }
  }
}
```

**4 points to explain:**
* **`task_data` does not reach the bidders.** Only `bid_job_description` is forwarded, so the RFP URL is put in both.
* **Bidders: named + topic.** The script prints how many resolved and the mode the job got.
  * 5 bidders: the topic join worked
  * 3 bidders: it did not
  * Mode "not reported": OpenArcade is older than bid job modes, so the job runs as `closed`
* **Two ways to hear of a job.**
  * `bid_request`: OpenArcade sends it to every company on the roster
  * Polling: UltraVideoTech and VideoProcTech also ask OpenArcade for jobs with their topic
  * A job is claimed once, so a company never bids twice
  * If a `bid_request` is lost, polling picks the job up after 30 seconds
* **No timeout in XChange.**
  * A subject that never bids leaves the task stuck in `bidding`
  * So every agent path must submit a bid

Submitting returns straight away. Use the `task_id` to check `GET /tasks/{id}` and `GET /tasks/{id}/output`.

In [ ]:
%%bash
(
  cd ../deploy_run
  bash va_bidding_request.sh
)

### 8.2. What XChange kept

* `task_assignment_status`: `pending` then `bidding` then `completed`
* `task_metadata.bid_job_id`: the bid job XChange opened
* `task_output`: the winner message, with evaluator scores
* Winner name is not a field. Check the dashboard or the bid job task results.

In [ ]:
%%bash
(
  cd ../../..
  set -a; source .env; set +a

  # The newest bidding task is the round just run. This query returns every task and the
  # body runs to a few hundred KB, so give curl room -- too short a --max-time here just
  # yields an empty result.
  LATEST=$(curl -s --max-time 60 -X POST "$EXCHANGE_BASE_URL/tasks/query" -H 'Content-Type: application/json' -d '{"task_assignment_type": "bidding"}' |
           jq '(.data // .) | sort_by(.taski_creation_time) | last')

  echo "$LATEST" | jq '{task_id, created: .taski_creation_time,
                        status: .task_assignment_status,
                        bid_job_opened: .task_metadata.bid_job_id,
                        invited_by_id: (.task_metadata.task_assignment.bid_job_subject_ids // []),
                        topics: (.task_metadata.task_assignment.topics // [])}'

  echo "--- GET /tasks/<id>/output"
  TASK_ID=$(echo "$LATEST" | jq -r .task_id)
  RESULT='.task_output.ack.result.messages[0].message_data.job_data.result_data'
  curl -s --max-time 30 "$EXCHANGE_BASE_URL/tasks/$TASK_ID/output" |
    jq "(.data // .) | {task_assignment_status, reason: ${RESULT}.reason, scores: ${RESULT}.scores}"
)

## 9. Read the result in the dashboard

* Read-only (one "clear HIS" button)
* Not needed for the round to run

In [ ]:
%%bash
(
  cd ../deploy_run
  nohup bash run_streamlit_app.bash > streamlit.log 2>&1 &
  sleep 3
  echo "Dashboard starting in the background on VA_STREAMLIT_PORT (default 8008)"
)

### What to show in each tab

* **Overview** - all bids in one table
  * **Decline Stage**: which agent stopped the bid
  * **Why not**: the reason for that stage
* **Companies** - each company's 6 agents, what each received and produced (live from HIS)
* **Timeline** - Gantt chart of the round
  * One row per agent, timing to the millisecond
  * Company dropdown filters to one vendor
* **Evaluation** - the 4 scores per company, plus what PQT and the evaluator received and decided
  * PQT: check table (required, presented, pass or fail)
  * Evaluator: bids seen, weights, every score, exclusions, tie-break
* **Raw** - task, bid job, bids and HIS records

## 10. How the winner is chosen

4 scores, each out of 100. Total out of 400. No weights between them.

| Score | How it is calculated |
| :--- | :--- |
| **Budget** | Ranked: lower total budget scores higher |
| **Sizing** | Ranked: weighted across CPU, RAM, disk, GPU |
| **Compliance** | `met / total`, same total for every company |
| **Endpoint** | `correct / total` from the live test against ground truth |

* Declined and PQT-rejected bids are removed before ranking
* Demo point: winning the live test does not always win the round

## 11. Run the checks without a cluster

* `local_check.py` runs the agent code without a cluster
* Quick way to test a config change before rebuilding 30 images
* The 4 function test suites run the same way

In [ ]:
%%bash
(
  cd ..
  set -a; source ../../.env; set +a
  PYTHONNOUSERSITE=1 PYTHONPATH="${AGENTS_SDK_PATH:-$HOME/Downloads/agent_codes}" \
    ../../venv/bin/python nodes/local_check.py | tail -20
)

In [ ]:
%%bash
(
  cd ../functions
  for f in va-bidding-pqt va-bid-eval va-rfp-requirements va-usecase-endpoint; do
    # stderr is suppressed on purpose: several of these tests drive a failure path and
    # the logging they provoke is the test working, not the function breaking.
    printf '  %-24s ' "$f"
    ( cd "$f" && ../../../../venv/bin/python test.py 2>/dev/null | grep -c '  ok  ' )
  done
)

## 12. Things that can go wrong in a live demo

* **Function re-uploaded but old code still runs:** bump the version. `delete.sh` removes old ones.
* **Warmup finishes in 1 second:** the old pod is still running. Check its age in the `policies` namespace.
* **Only 3 bidders:** `register_subjects_in_Exchange.sh` was not run after `register.sh`
* **Evaluator times out:** `warmup.sh` was skipped (60s, no retry)

## 13. Clean Up and Destroy Resources

### Remove the agent pods

In [ ]:
%%bash
(
  cd ../deploy_run
  bash deploy_remove.sh
)

### Remove the functions and their deployments

In [ ]:
%%bash
(
  cd ../functions
  bash delete.sh
)

### Unregister the 30 agent specs and the 5 exchange subjects

In [ ]:
%%bash
(
  cd ../spec
  bash unregister.sh
)